# 02 — Train HCCS Scorer

Trains the Hallucination Context Contrast Scorer (HCCS) using the contrastive
triplets generated in notebook 01.

**What it produces:** `checkpoints/hccs_best.pt` — trained MLP scorer weights.

**Pipeline:**
1. Load triplets from `data/triplets.jsonl`
2. Pre-compute CodeBERT embeddings for all queries and contexts (saved as `.npy`)
3. Train the MLP scorer with InfoNCE loss
4. Validate: scorer should rank positive > negative > 80% of the time
5. Save the best checkpoint to Drive

**Runtime:** ~minutes on T4 GPU (~0.5 CU)

---
Run `01_data_pipeline.ipynb` first to generate `data/triplets.jsonl`.

## Setup

In [ ]:
!pip install torch transformers numpy tqdm --quiet

In [ ]:
import sys
sys.path.insert(0, '..')

from notebooks.utils import mount_drive, check_gpu, save_checkpoint, load_checkpoint

my_drive = mount_drive()
DEVICE = check_gpu()

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

from haluguard.data_pipeline import load_triplets
from haluguard.hccs import HCCSScorer, batch_embed, embed_code, infonce_loss
from notebooks.utils import get_drive_path

DRIVE_DIR = get_drive_path('HaluGuard')
TRIPLETS_PATH = DRIVE_DIR / 'data' / 'triplets.jsonl'
EMB_DIR = DRIVE_DIR / 'data' / 'embeddings'
CKPT_DIR = DRIVE_DIR / 'checkpoints'
EMB_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Triplets: {TRIPLETS_PATH}')
print(f'Embeddings dir: {EMB_DIR}')
print(f'Checkpoints dir: {CKPT_DIR}')

## 1. Load Triplets

In [ ]:
triplets = load_triplets(TRIPLETS_PATH)
print(f'Loaded {len(triplets)} triplets')

# Train / val split (90 / 10)
import random
random.seed(42)
random.shuffle(triplets)
split = int(0.9 * len(triplets))
train_triplets = triplets[:split]
val_triplets   = triplets[split:]
print(f'Train: {len(train_triplets)}  Val: {len(val_triplets)}')

## 2. Pre-compute CodeBERT Embeddings

In [ ]:
ENCODER_NAME = 'microsoft/codebert-base'
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
encoder.eval()
print(f'CodeBERT loaded on {DEVICE}')

In [ ]:
# Pre-compute embeddings for all unique texts in the triplet set.
# This avoids re-running CodeBERT every epoch.

QUERY_EMB_PATH = EMB_DIR / 'query_embs.npy'
POS_EMB_PATH   = EMB_DIR / 'pos_embs.npy'
NEG_EMB_PATH   = EMB_DIR / 'neg_embs.npy'

if QUERY_EMB_PATH.exists() and POS_EMB_PATH.exists() and NEG_EMB_PATH.exists():
    print('Loading cached embeddings...')
    query_embs = np.load(QUERY_EMB_PATH)
    pos_embs   = np.load(POS_EMB_PATH)
    neg_embs   = np.load(NEG_EMB_PATH)
else:
    print('Computing embeddings (this may take a few minutes)...')
    queries   = [t.query            for t in triplets]
    pos_texts = [t.positive_context for t in triplets]
    neg_texts = [t.negative_context for t in triplets]

    query_embs = batch_embed(queries,   tokenizer, encoder, device=DEVICE)
    pos_embs   = batch_embed(pos_texts, tokenizer, encoder, device=DEVICE)
    neg_embs   = batch_embed(neg_texts, tokenizer, encoder, device=DEVICE)

    np.save(QUERY_EMB_PATH, query_embs)
    np.save(POS_EMB_PATH,   pos_embs)
    np.save(NEG_EMB_PATH,   neg_embs)
    print('Embeddings saved.')

print(f'Shapes — queries: {query_embs.shape}, pos: {pos_embs.shape}, neg: {neg_embs.shape}')

## 3. Train the MLP Scorer

In [ ]:
class TripletDataset(Dataset):
    """PyTorch Dataset wrapping pre-computed triplet embeddings."""

    def __init__(self, indices, query_embs, pos_embs, neg_embs):
        self.indices = indices
        self.query_embs = query_embs
        self.pos_embs   = pos_embs
        self.neg_embs   = neg_embs

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return (
            torch.tensor(self.query_embs[idx], dtype=torch.float32),
            torch.tensor(self.pos_embs[idx],   dtype=torch.float32),
            torch.tensor(self.neg_embs[idx],   dtype=torch.float32),
        )


train_idx = list(range(len(train_triplets)))
val_idx   = list(range(len(train_triplets), len(triplets)))

train_ds = TripletDataset(train_idx, query_embs, pos_embs, neg_embs)
val_ds   = TripletDataset(val_idx,   query_embs, pos_embs, neg_embs)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
# Training hyperparameters
EPOCHS   = 15
LR       = 1e-3
TAU      = 0.07

scorer = HCCSScorer(input_dim=1536, hidden_dim=256, dropout=0.1).to(DEVICE)
optimizer = torch.optim.AdamW(scorer.parameters(), lr=LR)

best_val_acc = 0.0
train_losses = []

for epoch in range(1, EPOCHS + 1):
    # --- Training ---
    scorer.train()
    epoch_loss = 0.0
    for q_emb, pos_emb, neg_emb in train_loader:
        q_emb   = q_emb.to(DEVICE)
        pos_emb = pos_emb.to(DEVICE)
        neg_emb = neg_emb.to(DEVICE)

        pos_input = torch.cat([q_emb, pos_emb], dim=1)  # (B, 1536)
        neg_input = torch.cat([q_emb, neg_emb], dim=1)

        pos_score = scorer(pos_input)  # (B, 1)
        neg_score = scorer(neg_input)

        loss = infonce_loss(pos_score, neg_score, tau=TAU)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    mean_loss = epoch_loss / len(train_loader)
    train_losses.append(mean_loss)

    # --- Validation ---
    scorer.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for q_emb, pos_emb, neg_emb in val_loader:
            q_emb   = q_emb.to(DEVICE)
            pos_emb = pos_emb.to(DEVICE)
            neg_emb = neg_emb.to(DEVICE)
            pos_score = scorer(torch.cat([q_emb, pos_emb], dim=1))
            neg_score = scorer(torch.cat([q_emb, neg_emb], dim=1))
            correct += (pos_score > neg_score).sum().item()
            total   += q_emb.size(0)

    val_acc = correct / max(total, 1)
    print(f'Epoch {epoch:2d}/{EPOCHS} | loss: {mean_loss:.4f} | val_acc: {val_acc:.3f}')

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(scorer, 'hccs_best.pt', drive_subdir='HaluGuard/checkpoints')
        print(f'  → New best ({val_acc:.3f}) saved')

print(f'\nTraining complete. Best val accuracy: {best_val_acc:.3f}')
print('Target: > 0.80 (80% of held-out triplets ranked correctly)')

## 4. Validate

In [ ]:
# Plot training loss curve
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('InfoNCE Loss')
    plt.title('HCCS Training Loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(CKPT_DIR / 'training_loss.png', dpi=150)
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot')

In [ ]:
# Qualitative check: score a few positive vs negative contexts
scorer.eval()

print('Sample scores (positive should be > negative):\n')
for i in range(min(5, len(val_triplets))):
    t = val_triplets[i]
    q_emb   = torch.tensor(query_embs[split + i], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    pos_emb = torch.tensor(pos_embs[split + i],   dtype=torch.float32).unsqueeze(0).to(DEVICE)
    neg_emb = torch.tensor(neg_embs[split + i],   dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pos_s = scorer(torch.cat([q_emb, pos_emb], dim=1)).item()
        neg_s = scorer(torch.cat([q_emb, neg_emb], dim=1)).item()

    status = '✓' if pos_s > neg_s else '✗'
    print(f'  [{status}] pos={pos_s:.3f}  neg={neg_s:.3f}  type={t.hallucination_type}')